# 03 · Analysis
## Measuring roughness and feature shape

Once data is levelled (notebook 02), the next step is turning pixels into numbers.
We cover two flavours of analysis that come up constantly in AFM, whatever your sample:

- **Whole-image roughness**: a single number (or two) summarising how rough a surface is.
- **Per-feature measurement**: segmenting individual objects (molecules, grains, domains)
  and measuring their size and shape.

Along the way we introduce some of the most useful general-purpose tools for this kind of
work: `scikit-image` for segmentation and shape measurement, `pandas` for organising
results into tables, and `scipy.stats` for fitting distributions.

| sample | analysis | tools introduced |
|---|---|---|
| DLC film | roughness (Sq, Sa) | plain NumPy |
| DNA plasmids (from nb 02) | segmentation + measurement table | `skimage.measure`, `pandas` |
| DNA minicircles | visualising + comparing feature shape | `scipy.stats`, more `skimage` |

**Contents**
1. Imports and tools (reused from notebook 02)
2. Surface roughness: Sq and Sa
3. Segmenting and measuring features: DNA plasmids
4. Visualising and comparing feature shape: DNA minicircles
5. Saving results

---

## 1 · Imports and tools

The masking and levelling functions are the same ones from notebook 02 — copied here so
this notebook stands on its own. See notebook 02 for how they're built; here we just use
them.

In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from matplotlib.colors import Normalize
from sklearn.linear_model import LinearRegression
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops_table
from skimage.segmentation import clear_border
from skimage.color import label2rgb

from AFMReader.general_loader import LoadFile
import playnano  # registers afm_brown / playnano_gold colormaps on import — not used otherwise

DATA_DIR    = Path(".") / "data" / "samples"
ARRAY_DIR   = Path(".") / "data" / "arrays"
RESULTS_DIR = Path(".") / "data" / "results"

print("Ready.")


In [ ]:

# --- Same tools as notebook 02 ---
def fit_plane(data, mask=None):
    """Return the best-fit plane. Fit background only if mask given (True=feature)."""
    h, w = data.shape
    Y, X = np.mgrid[0:h, 0:w]
    coords = np.column_stack([X.ravel(), Y.ravel()])
    z = data.ravel()
    keep = np.ones(z.shape, bool) if mask is None else ~mask.ravel()
    model = LinearRegression().fit(coords[keep], z[keep])
    return model.predict(coords).reshape(h, w)

def row_median_align(data, mask=None):
    """Subtract each row's median (background pixels only, if mask given)."""
    out = data.copy()
    for i in range(data.shape[0]):
        ref = data[i, :] if mask is None else data[i, ~mask[i, :]]
        if ref.size == 0:
            ref = data[i, :]
        out[i, :] -= np.median(ref)
    return out

def threshold_mask(data, k=1.0):
    """Boolean mask: True where height > mean + k*std (True = feature)."""
    return data > (data.mean() + k * data.std())

def show(img, title, cmap="playnano_gold", zmin=None, zmax=None, ax=None, colorbar=True):
    ax = ax or plt.gca()
    im = ax.imshow(img, cmap=cmap, vmin=zmin, vmax=zmax, origin="lower")
    ax.set_title(title); ax.axis("off")
    if colorbar:
        plt.colorbar(im, ax=ax, label="Height (nm)", fraction=0.046)

print("Tools ready.")


## 2 · Surface roughness: Sq and Sa

Two standard roughness numbers, both just statistics of the height distribution:

- **Sq** (RMS roughness): root-mean-square deviation from the mean height.
  `Sq = sqrt(mean((z - mean(z))**2))`
- **Sa** (average roughness): mean *absolute* deviation from the mean height.
  `Sa = mean(abs(z - mean(z)))`

Sq is the more commonly quoted of the two (it's what Gwyddion and most AFM software
report by default), but both are one line of NumPy.


In [ ]:

def Sq(data, mask=None):
    """Root-mean-square roughness. Excludes masked (True) pixels if given."""
    vals = data if mask is None else data[~mask]
    return np.sqrt(np.mean((vals - vals.mean())**2))

def Sa(data, mask=None):
    """Mean absolute roughness. Excludes masked (True) pixels if given."""
    vals = data if mask is None else data[~mask]
    return np.mean(np.abs(vals - vals.mean()))

img_dlc, px_dlc = LoadFile(DATA_DIR / "DLC.F 20um.000.spm", channel="Height").load()
img_dlc = img_dlc.astype(np.float64)

show(img_dlc, "DLC film (raw)")
plt.show()

print(f"Raw Sq:  {Sq(img_dlc):.3f} nm   (no levelling — tilt inflates this)")


### Levelling first

Roughness is meaningless on tilted data — tilt itself looks like "roughness" to Sq/Sa.
Level with a plane (the standard first step) and compare against the Gwyddion reference,
which used exactly the same single step (plane fit, no further correction).

In [ ]:

levelled_dlc = img_dlc - fit_plane(img_dlc)

print(f"Sq: {Sq(levelled_dlc):.3f} nm   (Gwyddion: 11.901 nm)")
print(f"Sa: {Sa(levelled_dlc):.3f} nm   (Gwyddion: 8.910 nm)")

show(levelled_dlc, "DLC film (plane-levelled)")
plt.show()


Our from-scratch plane fit agrees with Gwyddion to several significant figures;
useful confirmation that a hand-rolled NumPy/scikit-learn fit does the same job as an
established AFM analysis tool that you might have used before.


### Masked vs unmasked

Roughness isn't a fixed property of an image; it depends on *which pixels* you include.
A few tall contaminants or grains can dominate Sq even if the rest of the surface is very
flat. Mask them out and see how much the number changes. In this sample there are a
number of clear surface 'blobs' — we can mask these and calculate the roughness of the
underlying surface.

In [ ]:

mask_dlc = threshold_mask(levelled_dlc, k=2.5)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
show(levelled_dlc, "Levelled", ax=ax[0])
ax[1].imshow(mask_dlc, cmap="gray", origin="lower")
ax[1].set_title(f"Mask (covers {100*mask_dlc.mean():.1f}%)")
ax[1].axis("off")
plt.tight_layout(); plt.show()

print(f"Sq, unmasked: {Sq(levelled_dlc):.3f} nm")
print(f"Sq, masked:   {Sq(levelled_dlc, mask=mask_dlc):.3f} nm")
print(f"Sa, unmasked: {Sa(levelled_dlc):.3f} nm")
print(f"Sa, masked:   {Sa(levelled_dlc, mask=mask_dlc):.3f} nm")


**Takeaway:** always state whether a quoted roughness value is masked, and what was
excluded, and which steps were applied (plane only? row-aligned? masked?). "Sq = 2 nm"
means something different depending on all three.

## 3 · Segmenting and measuring features: DNA plasmids

We reload the levelled plasmid image from notebook 02; no need to repeat the levelling work.

It's worth **re-masking** the data at this point. For levelling, we wanted to mask every
non-background pixel; for analysis, we're often only interested in a subset (or a
slightly different threshold gives cleaner object boundaries). The levelling mask and
the analysis mask don't have to be the same one.

In [ ]:

raw_dna, px_dna = LoadFile(DATA_DIR / "20240513_SC_PJIP37_unknot_4ng_eph_ni__.0_00019.spm", channel="Height").load()
raw_dna = raw_dna.astype(np.float64)

# Leveling

levelled_dna = row_median_align(raw_dna - fit_plane(raw_dna))

dna_mask = threshold_mask(levelled_dna, k=2.5)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
show(levelled_dna, "DNA plasmids (levelled)", ax=ax[0])
show(dna_mask, "DNA mask", ax=ax[1])
plt.show()


### 3.1 · Labelling connected regions

`skimage.measure.label` turns a boolean mask into an integer image, each connected
group of `True` pixels gets its own ID. This is the segmentation step: from here on, each
labelled blob is a separate object we can measure.

In [ ]:

labeled_dna = label(dna_mask)
n_found = labeled_dna.max()
print(f"Found {n_found} connected regions (including any noise specks).")

plt.imshow(labeled_dna, cmap="nipy_spectral", origin="lower")
plt.title(f"Labelled regions ({n_found} found)")
plt.axis("off")
plt.colorbar(label="label ID", fraction=0.046)
plt.show()


### 3.2 · Measuring shape with `regionprops_table`

`regionprops_table` measures every labelled region in one call and returns a dict ready
for `pandas` — no manual looping needed. We pull geometric properties:

- `area`: region mask size in pixels
- `axis_major_length` / `axis_minor_length`: best-fit ellipse axes (good for rod-like shapes)
- `eccentricity`: 0 for a circle, close to 1 for an elongated shape
- `feret_diameter_max`: the largest end-to-end distance, regardless of shape (the
  "calliper diameter"); more honest than the major axis for irregular or looped shapes
- `orientation`, `centroid`: needed later to draw the measured axes on the image

In [ ]:

props = regionprops_table(
    labeled_dna,
    properties=("label", "area", "axis_major_length", "axis_minor_length",
                "eccentricity", "feret_diameter_max", "orientation", "centroid"),
)
df_dna = pd.DataFrame(props)
df_dna = df_dna.sort_values(by="area", ascending=False)
print(f"{len(df_dna)} regions before filtering, ordered by area:")
df_dna.head()


### 3.3 · Per-feature height

`regionprops_table` measures the shape of masks and doesn't handle height values. For height, we index the original levelled image with each region's own pixel mask; the same masking idea from notebook 02, just applied per-object instead of to the whole image.

In [ ]:

mean_heights, median_heights, max_heights = [], [], []

for lbl in df_dna["label"]:
    region_pixels = levelled_dna[labeled_dna == lbl]
    mean_heights.append(region_pixels.mean())
    median_heights.append(np.median(region_pixels))
    max_heights.append(region_pixels.max())

df_dna["height_mean_nm"]   = mean_heights
df_dna["height_median_nm"] = median_heights
df_dna["height_max_nm"]    = max_heights
df_dna[["label", "area", "height_mean_nm", "height_median_nm", "height_max_nm"]].head()


### 3.4 · Filtering out noise

A handful of regions are probably single-pixel noise specks, not real molecules. Filter
the **table**, not the image — much simpler than relabelling pixels.

In [ ]:

min_area_px = 100
n_before = len(df_dna)
df_dna = df_dna[df_dna["area"] >= min_area_px].reset_index(drop=True)
print(f"Kept {len(df_dna)} of {n_before} regions after filtering (area >= {min_area_px} px).")

# Only show regions that survived filtering — zero out anything else.
kept_labels = set(df_dna["label"])
filtered_labels = np.where(np.isin(labeled_dna, list(kept_labels)), labeled_dna, 0)
zmin, zmax = -2.0, 5.0

norm = Normalize(vmin=zmin, vmax=zmax)
img_rgb = plt.get_cmap("afm_brown")(norm(levelled_dna))[..., :3]   # drop alpha channel

# Create an overlay with the labeled masks
overlay = label2rgb(filtered_labels, image=img_rgb, bg_label=0, alpha=0.4, saturation=1)

plt.figure(figsize=(6, 6))
plt.imshow(overlay, origin="lower")
plt.title(f"{len(df_dna)} regions kept (coloured overlay)")
plt.axis("off")
plt.show()


### 3.5 · Converting to real units

Lengths scale with `pixel_to_nm`; **areas scale with its square**.

In [ ]:

df_dna["major_nm"]  = df_dna["axis_major_length"] * px_dna
df_dna["minor_nm"]  = df_dna["axis_minor_length"] * px_dna
df_dna["feret_nm"]  = df_dna["feret_diameter_max"] * px_dna
df_dna["area_nm2"]  = df_dna["area"] * px_dna**2
df_dna["aspect_ratio"] = df_dna["major_nm"] / df_dna["minor_nm"]

df_dna[["label", "major_nm", "minor_nm", "feret_nm", "aspect_ratio",
       "height_mean_nm", "height_median_nm", "height_max_nm"]].round(2)


That's the measurement table for the plasmids — `df_dna`. We'll come back to it
in Section 4 to compare against the minicircles.

## 4 · Visualising and comparing feature shape: DNA minicircles

The supercoiled DNA plasmids above are elongated by writhe, so major/minor axis length made good sense for them. DNA **minicircles** are small closed loops that are less able to deform and become elongated, "length and width" is a less natural description for these than for a rod. Better metrics for loop/circular shapes:

- **`eccentricity`**: should sit near 0 (a near-circular object), in contrast to a  rod's eccentricity near 1.
- **`feret_diameter_max`**: the overall span of the loop, regardless of its (possibly irregular) shape; more robust than an ellipse-based axis length for a ring.
- **`aspect_ratio`** (major/minor): should sit close to 1 for a circle, vs. a rod's much larger value.

The loading and levelling are filled in below (revision from notebook 02). The segmentation and measurement steps follow exactly the same pattern as Section 3.

In [ ]:

img_mini, px_mini = LoadFile(DATA_DIR / "20190201_339_-2_Ni_3mM_HEPES_20mM.0_00025.spm", channel="Height").load()
img_mini = img_mini.astype(np.float64)

rough_mini = row_median_align(img_mini - fit_plane(img_mini))
mask_mini  = threshold_mask(rough_mini, k=1.8)
levelled_mini = row_median_align(img_mini - fit_plane(img_mini, mask=mask_mini),
                                 mask=mask_mini)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
show(levelled_mini, "DNA minicircles (levelled)", ax=ax[0])
ax[1].imshow(mask_mini, cmap="gray", origin="lower")
ax[1].set_title("Mask"); ax[1].axis("off")
plt.tight_layout()
plt.show()


### 4.1 · Segment and measure (same steps as Section 3)

In [ ]:

labeled_mini = label(mask_mini)
print(f"Found {labeled_mini.max()} connected regions.")

# Remove any region touching the frame edge — a scan boundary cuts these off
# arbitrarily, so their area, axis lengths, and eccentricity describe a fragment,
# not the real feature. Better to drop them than measure something misleading.
labeled_mini = clear_border(labeled_mini)
n_remaining = len(np.unique(labeled_mini)) - 1   # -1 excludes the background label (0)
print(f"{n_remaining} regions remain after removing edge-touching objects.")

props_mini = regionprops_table(
    labeled_mini,
    properties=("label", "area", "axis_major_length", "axis_minor_length",
                "eccentricity", "feret_diameter_max", "orientation", "centroid"),
)
df_mini = pd.DataFrame(props_mini)
min_area_px_mini = 500
max_area_px_mini = 1500
df_mini = df_mini[df_mini["area"] >= min_area_px_mini].reset_index(drop=True)
df_mini = df_mini[df_mini["area"] < max_area_px_mini].reset_index(drop=True)

print(f"{len(df_mini)} regions kept after area filtering.")

df_mini["feret_nm"]     = df_mini["feret_diameter_max"] * px_mini
df_mini["aspect_ratio"] = df_mini["axis_major_length"] / df_mini["axis_minor_length"]
heights_mean, heights_max = [], []
for lbl in df_mini["label"]:
    px_vals = levelled_mini[labeled_mini == lbl]
    heights_mean.append(px_vals.mean())
    heights_max.append(px_vals.max())
df_mini["height_mean_nm"] = heights_mean
df_mini["height_max_nm"]  = heights_max
df_mini[["label", "feret_nm", "aspect_ratio", "eccentricity",
        "height_mean_nm", "height_max_nm"]].round(2)


### 4.2 · Visualise: axes *and* eccentricity together

We draw the same major/minor axis lines as before. For a near-circular loop, the blue
and red lines should come out close to the same length — a visual echo of "eccentricity
near 0". We annotate each region with its actual eccentricity so the picture and the
number reinforce each other.

In [ ]:

fig, ax = plt.subplots(figsize=(6, 6))
show(levelled_mini, f"DNA minicircles — {len(df_mini)} loops measured", ax=ax)

for _, region in df_mini.iterrows():
    y0, x0 = region["centroid-0"], region["centroid-1"]
    orientation = region["orientation"]

    x1 = x0 + np.sin(orientation) * region["axis_major_length"] / 2
    y1 = y0 + np.cos(orientation) * region["axis_major_length"] / 2
    x2 = x0 - np.sin(orientation) * region["axis_major_length"] / 2
    y2 = y0 - np.cos(orientation) * region["axis_major_length"] / 2
    ax.plot((x1, x2), (y1, y2), "-b", linewidth=1.5)

    x3 = x0 + np.cos(orientation) * region["axis_minor_length"] / 2
    y3 = y0 - np.sin(orientation) * region["axis_minor_length"] / 2
    x4 = x0 - np.cos(orientation) * region["axis_minor_length"] / 2
    y4 = y0 + np.sin(orientation) * region["axis_minor_length"] / 2
    ax.plot((x3, x4), (y3, y4), "-r", linewidth=1.5)

    ax.text(x0, y0, f"e={region['eccentricity']:.2f}", color="yellow",
            fontsize=8, ha="center", va="center")

plt.tight_layout(); plt.show()


### 4.3 · Distributions: size and shape

Two histograms: **size** (Feret diameter, with a fitted normal distribution overlaid —
the same `scipy.stats.norm.fit` trick as before) and **shape** (eccentricity, plotted
plain since it's bounded between 0 and 1 and a normal fit there wouldn't mean much).

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

feret_vals = df_mini["feret_nm"].values
mu, sigma = stats.norm.fit(feret_vals)
axes[0].hist(feret_vals, bins=10, density=True, alpha=0.6, label="measured")
xs = np.linspace(feret_vals.min(), feret_vals.max(), 200)
axes[0].plot(xs, stats.norm.pdf(xs, mu, sigma), "r-", lw=2,
            label=f"normal fit (μ={mu:.1f}, σ={sigma:.1f} nm)")
axes[0].set_xlabel("Feret diameter (nm)"); axes[0].set_ylabel("Density")
axes[0].set_title("Minicircle size"); axes[0].legend()

axes[1].hist(df_mini["eccentricity"], bins=10, color="orange", alpha=0.7)
axes[1].set_xlabel("Eccentricity"); axes[1].set_ylabel("Count")
axes[1].set_title("Minicircle shape (0 = circle, 1 = line)")
axes[1].set_xlim(0, 1)

plt.tight_layout(); plt.show()

summary_mini = df_mini[["feret_nm", "aspect_ratio", "eccentricity",
                        "height_mean_nm", "height_max_nm"]].agg(["mean", "std"])
print(summary_mini.round(3))


### 4.4 · Same code, different shape: plasmids vs minicircles

Both samples were measured with the *exact same* `regionprops_table` call; only which columns we pay attention to changes, because the feature geometry is different.

In [ ]:

comparison = pd.DataFrame({
    "DNA plasmids (elongated)":     [df_dna["aspect_ratio"].mean(), df_dna["eccentricity"].mean()],
    "DNA minicircles (round)": [df_mini["aspect_ratio"].mean(), df_mini["eccentricity"].mean()],
}, index=["mean aspect ratio", "mean eccentricity"])

print(comparison.round(3))
print()
print("The supercoiled plasmids come out more elongated than minicircles on both metrics, as expected")
print("for a rod compared to a loop. But the minicircles aren't close to a perfect")
print("circle either — real loops are rarely perfectly round: they can be supercoiled,")
print("relaxed into an oval.")


### Your turn

Try this whole pipeline on your own data:

1. Load your image with `LoadFile(...)`.
2. Level it with `fit_plane` / `row_median_align` / `fit_polynomial` (notebook 02).
3. Mask, `label`, `regionprops_table`, filter by area, convert to nm.
4. Pick the metrics that actually suit *your* feature's shape: rod, loop, blob, or something else entirely.

In [ ]:

# Your turn:
# your_img, your_px = LoadFile(...).load()
# ... level, mask, label, regionprops_table, filter, convert ...


## 5 · Saving results

Save both measurement tables as CSVs — a natural hand-off to notebook 04, which covers
exporting figures and results for publication.

In [ ]:

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df_dna.to_csv(RESULTS_DIR / "plasmid_measurements.csv", index=False)
df_mini.to_csv(RESULTS_DIR / "minicircle_measurements.csv", index=False)
print("Saved plasmid_measurements.csv and minicircle_measurements.csv")

for fname in ("plasmid_measurements.csv", "minicircle_measurements.csv"):
    check = pd.read_csv(RESULTS_DIR / fname)
    print(f"{fname}: {len(check)} rows, columns: {list(check.columns)}")
    

---

### Recap

- **Sq / Sa** are one-line NumPy statistics of height — but always level first, and state
  whether the value is masked.
- **`skimage.measure.label`** turns a mask into separate numbered objects;
  **`regionprops_table`** measures them all at once, ready for a `pandas.DataFrame`.
- Filter noise by filtering the **table** (`area >= threshold`), not by relabelling pixels.
- Per-feature height comes from indexing the original array with each region's own mask —
  the same idea as notebook 02's background masking, applied per-object.
- The *same* segmentation and measurement code works on any shape — choose which columns
  to interpret based on the feature: major/minor axis for rods, `eccentricity` /
  `feret_diameter_max` for loops or irregular shapes.

**Next:** `04_export.ipynb` — publication-ready figures, scale bars, and exporting results.